| Group | AI Assisted Reporting | Outcome  |
|-------|-----------------------|----------|
| 1     | No                    | Positive |
| 2     | No                    | Negative |
| 3     | Yes                   | Positive |
| 4     | Yes                   | Negative |
| 5     | Yes + GenAI           | Positive |
| 6     | Yes + GenAI           | Negative |

In [1]:
import polars as pl
import pandas as pd
import pingouin as pg
from scipy.stats import mannwhitneyu

from pyprojroot import here

root_dir = here()

In [2]:
construct_items = {
    "OA": [
        "OA1",
        "OA2",
        "OA3",
        "OA4",
    ],
    "PA": [
        "PA1",
        "PA2",
        "PA3",
        "PA4",
    ],
    "ITU": [
        "IUT1",
        "IUT2",
        "IUT3",
    ],
    "PRIV": [
        "PRIV1",
        "PRIV2",
        "PRIV3",
    ],
    "RA": [
        "RA1",
        "RA2",
        "RA3",
    ],
    "RC": [
        "RC1",
        "RC2",
        "RC3",
    ],
}


In [3]:
# df_raw = pl.read_excel(
#     root_dir / "data" / "codebook_ntnu_survey_2026-05-14_14-00.xlsx", sheet_name="COMPLETE DATASET"
# )

df_raw = pl.from_pandas(
    pd.read_csv(
        root_dir / "data" / "soci" / "0-300" / "cleaned.csv", encoding="ISO-8859-1"
    )
)

In [4]:
df_raw

CASE,SERIAL,REF,QUESTNNR,MODE,STARTED,FEEDBACK,FT02_01,VOCAB,OA2,OA4,RA2,PA1,RA1,RC1,PA3,PRIV1,IUT2,RA3,CTRL1,OA3,PA4,RC3,PRIV2,PA2,RC2,IUT1,PRIV3,OA1,IUT3,MARK1,MARK2,RISK,PAKNOW,RD01_CP,GROUP,RD02_CP,RD02,ATT_SC,mc1,MC2,TIME001,TIME002,TIME003,TIME004,TIME005,TIME007,TIME008,TIME009,TIME010,TIME011,TIME012,TIME014,TIME015,TIME016,TIME017,TIME018,TIME021,TIME022,TIME_SUM,MAILSENT,LASTDATA,STATUS,FINISHED,Q_VIEWER,LASTPAGE,MAXPAGE,MISSING,MISSREL,TIME_RSI
i64,f64,f64,str,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,i64,i64,i64,f64,str,f64,i64,i64,i64,i64,i64,i64,f64
193,null,null,"""affective_grid""","""interview""","""5/13/26 13:21""",null,"""5c3a4e759e794d000129a225""",1,2,6,6,6,6,5,2,6,4,6,2,6,5,6,6,5,2.0,5.0,2.0,6.0,5.0,2,6,4,1,0,6,0,4,1,6,2,4,2,2,30,110,6,5,8,11,10,5,19,32,14,15,25.0,15,2,315,null,"""5/13/26 13:26""",null,1,0,22,22,3,1,1.92
195,null,null,"""affective_grid""","""interview""","""5/13/26 13:21""",null,"""664b8bef90cdb799d4c4c35d""",1,4,5,6,5,7,6,3,4,6,6,2,7,4,6,5,4,6.0,6.0,7.0,6.0,7.0,5,5,5,1,0,3,1,4,1,6,6,8,14,52,48,132,5,7,29,8,18,4,15,13,8,10,11.0,14,7,403,null,"""5/13/26 13:28""",null,1,0,22,22,3,1,1.57
196,null,null,"""affective_grid""","""interview""","""5/13/26 13:21""",null,"""6a03322d84fdeca625f08127""",1,5,6,6,6,6,5,5,5,6,5,2,5,7,5,5,7,5.0,5.0,6.0,5.0,4.0,5,2,7,2,0,2,0,6,1,6,3,7,2,4,45,37,9,2,13,7,12,7,12,13,12,16,78.0,18,1,295,null,"""5/13/26 13:26""",null,1,0,22,22,3,1,1.91
197,null,null,"""affective_grid""","""interview""","""5/13/26 13:21""","""none""","""5d1b3627b6a81c0001f87c85""",1,5,6,5,5,6,5,5,5,5,5,2,5,5,6,6,5,5.0,5.0,5.0,5.0,6.0,5,5,6,1,0,1,0,1,1,5,6,3,3,8,6,24,4,15,16,5,7,4,10,14,8,8,13.0,16,203,172,null,"""5/13/26 13:27""",null,1,0,22,22,0,0,2.23
198,null,null,"""affective_grid""","""interview""","""5/13/26 13:21""",null,"""602539c6fe366c0008c815cb""",1,2,6,4,2,6,5,5,7,2,5,2,5,6,6,7,5,5.0,2.0,1.0,5.0,2.0,1,1,9,2,1,3,1,6,1,6,5,8,2,27,114,121,12,15,70,20,42,9,25,27,20,30,42.0,24,2,610,null,"""5/13/26 13:31""",null,1,0,22,22,3,1,1.05
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
642,null,null,"""affective_grid""","""interview""","""5/15/26 13:46""",null,"""69f5167144f21c71a573746f""",1,6,5,5,5,5,5,6,5,6,5,2,6,6,5,6,5,5.0,6.0,5.0,5.0,5.0,6,6,7,2,65,3,59,2,1,5,6,72,12,21,53,82,10,11,51,28,10,3,17,12,13,10,17.0,46,13,418,null,"""5/15/26 13:54""",null,1,0,22,22,3,1,1.25
650,null,null,"""affective_grid""","""interview""","""5/15/26 13:58""","""i was confused if i was me or …","""69ffb64219d501a81d7e40c6""",1,6,6,3,2,3,6,3,3,6,3,2,6,2,5,3,6,5.0,5.0,5.0,6.0,6.0,6,2,8,2,65,1,60,4,1,6,6,12,43,10,39,82,13,11,21,7,16,13,18,19,28,28,81.0,16,22,479,null,"""5/15/26 14:06""",null,1,0,22,22,0,0,1.06
651,null,null,"""affective_grid""","""interview""","""5/15/26 14:07""",null,"""69f3c082cc8660d8c11ee447""",1,6,7,5,6,5,6,2,6,6,5,2,5,6,6,6,7,6.0,6.0,2.0,6.0,6.0,7,6,9,3,65,4,60,5,1,5,1,12,3,21,40,73,29,25,28,22,34,48,35,43,32,35,36.0,49,19,584,null,"""5/15/26 14:17""",null,1,0,22,22,0,0,0.78


In [5]:
required_cols = [
    "ATT_SC",
    "CTRL1",
    "GROUP",
    "IUT1",
    "IUT2",
    "IUT3",
    "MARK1",
    "MARK2",
    "mc1",
    "MC2",
    "OA1",
    "OA2",
    "OA3",
    "OA4",
    "PA1",
    "PA2",
    "PA3",
    "PA4",
    "PAKNOW",
    "PRIV1",
    "PRIV2",
    "PRIV3",
    "RA1",
    "RA2",
    "RA3",
    "RC1",
    "RC2",
    "RC3",
    "RD01_CP",
    "RD02_CP",
    "RD02",
    "RISK",
]


In [6]:
reversed_items = ["PRIV3"]

In [7]:
def correct_reversed(df: pl.DataFrame, reversed_items: list):
    return df.with_columns([(8 - pl.col(c)).alias(c) for c in reversed_items])

In [8]:
df = (
    df_raw.filter(
        pl.col("FT02_01") != "MiriamTest",
    )
    .drop_nulls(subset=required_cols)
    .pipe(correct_reversed, reversed_items)
)

In [9]:
df.select(["TIME_SUM"]).with_row_index().plot.scatter(x="index", y="TIME_SUM")

alt.Chart(...)

In [10]:
df

CASE,SERIAL,REF,QUESTNNR,MODE,STARTED,FEEDBACK,FT02_01,VOCAB,OA2,OA4,RA2,PA1,RA1,RC1,PA3,PRIV1,IUT2,RA3,CTRL1,OA3,PA4,RC3,PRIV2,PA2,RC2,IUT1,PRIV3,OA1,IUT3,MARK1,MARK2,RISK,PAKNOW,RD01_CP,GROUP,RD02_CP,RD02,ATT_SC,mc1,MC2,TIME001,TIME002,TIME003,TIME004,TIME005,TIME007,TIME008,TIME009,TIME010,TIME011,TIME012,TIME014,TIME015,TIME016,TIME017,TIME018,TIME021,TIME022,TIME_SUM,MAILSENT,LASTDATA,STATUS,FINISHED,Q_VIEWER,LASTPAGE,MAXPAGE,MISSING,MISSREL,TIME_RSI
i64,f64,f64,str,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,i64,i64,i64,f64,str,f64,i64,i64,i64,i64,i64,i64,f64
193,null,null,"""affective_grid""","""interview""","""5/13/26 13:21""",null,"""5c3a4e759e794d000129a225""",1,2,6,6,6,6,5,2,6,4,6,2,6,5,6,6,5,2.0,5.0,6.0,6.0,5.0,2,6,4,1,0,6,0,4,1,6,2,4,2,2,30,110,6,5,8,11,10,5,19,32,14,15,25.0,15,2,315,null,"""5/13/26 13:26""",null,1,0,22,22,3,1,1.92
195,null,null,"""affective_grid""","""interview""","""5/13/26 13:21""",null,"""664b8bef90cdb799d4c4c35d""",1,4,5,6,5,7,6,3,4,6,6,2,7,4,6,5,4,6.0,6.0,1.0,6.0,7.0,5,5,5,1,0,3,1,4,1,6,6,8,14,52,48,132,5,7,29,8,18,4,15,13,8,10,11.0,14,7,403,null,"""5/13/26 13:28""",null,1,0,22,22,3,1,1.57
196,null,null,"""affective_grid""","""interview""","""5/13/26 13:21""",null,"""6a03322d84fdeca625f08127""",1,5,6,6,6,6,5,5,5,6,5,2,5,7,5,5,7,5.0,5.0,2.0,5.0,4.0,5,2,7,2,0,2,0,6,1,6,3,7,2,4,45,37,9,2,13,7,12,7,12,13,12,16,78.0,18,1,295,null,"""5/13/26 13:26""",null,1,0,22,22,3,1,1.91
197,null,null,"""affective_grid""","""interview""","""5/13/26 13:21""","""none""","""5d1b3627b6a81c0001f87c85""",1,5,6,5,5,6,5,5,5,5,5,2,5,5,6,6,5,5.0,5.0,3.0,5.0,6.0,5,5,6,1,0,1,0,1,1,5,6,3,3,8,6,24,4,15,16,5,7,4,10,14,8,8,13.0,16,203,172,null,"""5/13/26 13:27""",null,1,0,22,22,0,0,2.23
198,null,null,"""affective_grid""","""interview""","""5/13/26 13:21""",null,"""602539c6fe366c0008c815cb""",1,2,6,4,2,6,5,5,7,2,5,2,5,6,6,7,5,5.0,2.0,7.0,5.0,2.0,1,1,9,2,1,3,1,6,1,6,5,8,2,27,114,121,12,15,70,20,42,9,25,27,20,30,42.0,24,2,610,null,"""5/13/26 13:31""",null,1,0,22,22,3,1,1.05
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
642,null,null,"""affective_grid""","""interview""","""5/15/26 13:46""",null,"""69f5167144f21c71a573746f""",1,6,5,5,5,5,5,6,5,6,5,2,6,6,5,6,5,5.0,6.0,3.0,5.0,5.0,6,6,7,2,65,3,59,2,1,5,6,72,12,21,53,82,10,11,51,28,10,3,17,12,13,10,17.0,46,13,418,null,"""5/15/26 13:54""",null,1,0,22,22,3,1,1.25
650,null,null,"""affective_grid""","""interview""","""5/15/26 13:58""","""i was confused if i was me or …","""69ffb64219d501a81d7e40c6""",1,6,6,3,2,3,6,3,3,6,3,2,6,2,5,3,6,5.0,5.0,3.0,6.0,6.0,6,2,8,2,65,1,60,4,1,6,6,12,43,10,39,82,13,11,21,7,16,13,18,19,28,28,81.0,16,22,479,null,"""5/15/26 14:06""",null,1,0,22,22,0,0,1.06
651,null,null,"""affective_grid""","""interview""","""5/15/26 14:07""",null,"""69f3c082cc8660d8c11ee447""",1,6,7,5,6,5,6,2,6,6,5,2,5,6,6,6,7,6.0,6.0,6.0,6.0,6.0,7,6,9,3,65,4,60,5,1,5,1,12,3,21,40,73,29,25,28,22,34,48,35,43,32,35,36.0,49,19,584,null,"""5/15/26 14:17""",null,1,0,22,22,0,0,0.78


In [11]:
# check if ctl was answered correctly
assert df["CTRL1"].unique().item() == 2

In [12]:
# check if comprehension check was answered correctly
assert df["ATT_SC"].unique().item() == 1

In [13]:
df["GROUP"].value_counts().sort(by="GROUP").plot.bar(
    x="GROUP",
    y="count",
)

alt.Chart(...)

## Cronbachs Alpha

In [41]:
ca_threshold = 0.7


for construct, items in construct_items.items():
    ca = pg.cronbach_alpha(df[items].to_pandas())[0]
    if ca < ca_threshold:
        print(f"Cr. alpha too low for {construct}: {round(ca, 3)} ❌")
    else:
        print(f"Cr. alpha fine for {construct}: {round(ca, 3)} ✅")

Cr. alpha too low for OA: 0.516 ❌
Cr. alpha too low for PA: 0.624 ❌
Cr. alpha fine for ITU: 0.967 ✅
Cr. alpha too low for PRIV: 0.565 ❌
Cr. alpha fine for RA: 0.796 ✅
Cr. alpha too low for RC: 0.662 ❌


In [26]:
construct_items["OA"]

['OA1', 'OA2', 'OA3', 'OA4']

In [28]:
df['OA1', 'OA2', 'OA3', 'OA4'].mean()

OA1,OA2,OA3,OA4
f64,f64,f64,f64
5.180602,5.454849,4.675585,5.541806


In [29]:
df['OA1', 'OA2', 'OA3', 'OA4'].std()

OA1,OA2,OA3,OA4
f64,f64,f64,f64
1.376026,1.34631,1.706218,1.433337


In [39]:
df["OA1", "OA2", "OA3", "OA4"].with_columns(
    OA1 = pl.col("OA1").cast(pl.Int64)
).to_pandas()

,OA1,OA2,OA3,OA4
0,6,2,6,6
1,6,4,7,5
2,5,5,5,6
3,5,5,5,6
4,5,2,5,6
...,...,...,...,...
294,5,6,6,5
295,6,6,6,6
296,6,6,5,7
297,5,6,5,5


In [33]:
pg.cronbach_alpha(df[["OA1", "OA2", "OA3", "OA4"]].to_pandas())

(np.float64(0.5163780160290974), array([0.42, 0.6 ]))

## Manipulation Check

In [19]:
df.group_by("GROUP").agg(pl.col("mc1").std()).sort(by="GROUP")

GROUP,mc1
i64,f64
1,0.867497
2,1.377562
3,1.205036
4,1.07663
5,1.486301
6,0.828189


In [20]:
df.group_by("GROUP").agg(pl.col("mc1").mean()).sort(by="GROUP").plot.bar(
    x="GROUP", y="mc1"
)

alt.Chart(...)

In [21]:
# Group 1, no AI: [1,2]
# Group 2, yes AI: [3,4,5,6]

group1 = df.filter(pl.col("GROUP").is_in([1,2]))["mc1"].to_list()
group2 = df.filter(~pl.col("GROUP").is_in([1,2]))["mc1"].to_list()

In [22]:
_, p = mannwhitneyu(group1, group2)
p

np.float64(0.06284610586110151)

In [23]:
df.group_by("GROUP").agg(pl.col("MC2").mean()).sort(by="GROUP").plot.bar(
    x="GROUP", y="MC2"
)

alt.Chart(...)

In [24]:
# Group 1, positive outcome: 1,3,5
# Group 2, negative outcome: 2,4,6

group1 = df.filter(pl.col("GROUP").is_in([1,3,5]))["MC2"].to_list()
group2 = df.filter(~pl.col("GROUP").is_in([1,3,5]))["MC2"].to_list()

In [25]:
_, p = mannwhitneyu(group1, group2)
p

np.float64(6.345301366559297e-33)